# Pipeline Architecture — LangGraph

**Pattern:** ETL as graph nodes

```
START → extract_node → transform_node → load_node → END
```

Each stage is a graph node. State tracks what's been extracted, transformed, and loaded.

In [ ]:
import os, json
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
RAW = ["Tokyo|Clear|18|Low|10|22:30 JST", "Paris|Partly Cloudy|16|Low|8|15:30 CET", "Bangalore|Rainy|25|Medium|6|20:00 IST"]
print("✓ Ready")

In [ ]:
class PipeState(TypedDict):
    raw_records: list; extracted: Optional[list]; transformed: Optional[list]; final_report: Optional[str]

def extract_node(s):
    records = []
    for line in s["raw_records"]:
        c,w,t,sl,ss,tm = line.split("|")
        records.append({"city":c,"weather":w,"temp_c":int(t),"safety_level":sl,"safety_score":int(ss),"local_time":tm})
    print(f"  [extract] {len(records)} records")
    return {"extracted": records}

def transform_node(s):
    sm = {"Clear":9,"Partly Cloudy":7,"Rainy":6,"Cloudy":5}
    records = s["extracted"].copy()
    for r in records:
        r["weather_score"] = sm.get(r["weather"],5)
        r["combined_score"] = round((r["weather_score"]+r["safety_score"])/2,1)
    records.sort(key=lambda x:x["combined_score"],reverse=True)
    for i,r in enumerate(records,1): r["rank"] = i
    print("  [transform] scored+ranked")
    return {"transformed": records}

def load_node(s):
    r = llm.invoke([SystemMessage(content="Format into markdown travel ranking report."),
                    HumanMessage(content=f"Data: {json.dumps(s['transformed'],indent=2)}")])
    print("  [load] report done")
    return {"final_report": r.content}

builder = StateGraph(PipeState)
for name,fn in [("extract",extract_node),("transform",transform_node),("load",load_node)]:
    builder.add_node(name, fn)
builder.add_edge(START,"extract"); builder.add_edge("extract","transform")
builder.add_edge("transform","load"); builder.add_edge("load",END)
graph = builder.compile()
print("Pipeline graph built")

In [ ]:
result = graph.invoke({"raw_records":RAW,"extracted":None,"transformed":None,"final_report":None})
print(result["final_report"])

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Pure Python nodes in a graph | Not all nodes need to be LLM calls |
| `stream_mode="updates"` shows each stage | Pipeline stages are fully observable |
| State accumulates across stages | Each stage reads the previous stage's output from state |

### Next: [CrewAI Pipeline →](../CrewAI/pipeline.ipynb)